<a href="https://colab.research.google.com/github/PriyankaDevaprasad/Diamond_Price_Prediction/blob/main/notebook/Streamlit_Diamond.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Diamond Price Prediction**

The trained machine learning models are deployed using Streamlit to create an interactive web application for:

💎 Diamond Price Prediction – Predicts the estimated diamond price in INR.

📊 Market Segment Prediction – Identifies the diamond's market segment using K-Means clustering.

The application uses the saved models and preprocessing objects to generate predictions from user-provided diamond attributes.

In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.8 MB/s eta 0:00:00
--2026-09-23 14:00:09--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.9.1/cloudflared-linux-amd64 [following]
--2026-09-23 14:00:09--  https://github.com/cloudflare/cloudflared/releases/download/2026.9.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/15b79d32-3929-4df3-b051-cd9f5815b612?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-23T14%3A57%3A45Z&rscd=attachment%3B+filena

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import pickle
import numpy as np

with open("ordinal_encoder.pkl", 'rb')as file:
  ordinal=pickle.load(file)

with open("best_model_diamond.pkl", 'rb') as file:
  model=pickle.load(file)

cluster_columns = joblib.load("cluster_columns.pkl")
cluster_model=joblib.load("cluster_model.pkl")

cluster_scaler = joblib.load("scaler.pkl")
cluster_names=joblib.load("cluster_names.pkl")

st.set_page_config(page_title="Diamond Price Prediction",page_icon="💎",layout="wide")
st.title("💎 Diamond Price Predictor")
st.divider()
st.subheader("Enter the details of the diamond")
col1, col2 = st.columns(2)

with col1:
      carat = st.number_input("Carat")
      cut = st.selectbox("Cut",["Fair", "Good", "Very Good", "Premium", "Ideal"])
      color = st.selectbox("Color",["D", "E", "F", "G", "H", "I", "J"])
      clarity = st.selectbox("Clarity",["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1"])
with col2:
      x = st.number_input("Length (x) in mm")
      y = st.number_input("Width (y) in mm")
      z = st.number_input("Height (z) in mm")

st.divider()

col3, col4 = st.columns(2)
input_data = pd.DataFrame({'carat':[carat],'cut':[cut],'color':[color],'clarity':[clarity],'x':[x],'y':[y],'z':[z]})
ord_cols=['cut', 'color', 'clarity']
input_data[ord_cols] = ordinal.transform(input_data[ord_cols])

with col3:
    st.subheader("💰 Price Prediction")
    predict=st.button("Predict Price", type="primary",key="predict")
    if predict:
        model_data=['carat','x','y','color','clarity']
        prediction = model.predict(input_data[model_data])
        pred_price = np.expm1(prediction[0])
        predicted_price_inr = pred_price * 95.74
        st.success(f"Predicted Price: Rs. {predicted_price_inr:,.2f}")
with col4:
    st.subheader("📊 Diamond Cluster")
    cluster=st.button("Predict Cluster", type="primary",key="cluster")
    if cluster:
        input_data_cluster=input_data[cluster_columns]
        input_data_cluster=cluster_scaler.transform(input_data_cluster)
        cluster_pred=cluster_model.predict(input_data_cluster)
        st.success(f"Predicted Cluster: {cluster_names[cluster_pred[0]]}")



Overwriting app.py


In [ ]:
!streamlit run /content/app.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://manufactured-accessories-climate-wait.trycloudflare.com
